# Orbit Wars — 可视化对局
加载训练好的 checkpoint，跑一局并渲染动画。

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import torch
from kaggle_environments import make

from config import (
    CHECKPOINT_DIR, CHANNELS, BOARD_SIZE,
    RES_BLOCKS, RES_FILTERS, DEVICE,
    C_PUCT, MAX_GAME_STEPS, MCTS_SIMULATIONS,
)
from network import PolicyValueNetwork
from mcts import MCTS
import train as tm

In [ ]:
# ── 参数 ──────────────────────────────────────────────
CKPT_NAME   = "interrupt"   # latest / interrupt / iter_50
OPPONENT    = "random"      # "random" 或 "deepseek"
NUM_AGENTS  = 2             # 2 或 4
EVAL_SIMS   = min(48, MCTS_SIMULATIONS)
GAME_SEED   = 42
# ──────────────────────────────────────────────────────

In [ ]:
# 加载网络
ckpt_path = os.path.join(CHECKPOINT_DIR, f"{CKPT_NAME}.pt")
net = PolicyValueNetwork(CHANNELS, BOARD_SIZE, RES_BLOCKS, RES_FILTERS).to(DEVICE)
ck  = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
net.load_state_dict(ck["model_state_dict"])
net.eval()
print(f"已加载 {ckpt_path}  iteration={ck.get('iteration','?')}")

In [ ]:
# 加载对手 agent
if OPPONENT == "random":
    import importlib
    orb = importlib.import_module("kaggle_environments.envs.orbit_wars.orbit_wars")
    opp_fn = orb.random_agent
else:
    import importlib.util
    v1_path = os.path.normpath(os.path.join(
        os.path.dirname(os.path.abspath('__file__')), "..", "v1_rule", "v1_deepseek", "main.py"
    ))
    spec = importlib.util.spec_from_file_location("v1_deepseek_main", v1_path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    opp_fn = mod.agent

print(f"对手: {OPPONENT}")

In [ ]:
# 构建 net agent
HERO = 0   # net 占第几个槽位（0 or 1）

env = make(
    "orbit_wars",
    debug=True,
    configuration={"episodeSteps": MAX_GAME_STEPS, "seed": GAME_SEED},
)

def net_agent(obs, config):
    info = getattr(env, "info", None) or {}
    ep_seed = info.get("seed")
    cs = float(tm._read(config, "cometSpeed", 4.0) or 4.0)
    sp = float(tm._read(config, "shipSpeed",  tm.MAX_SPEED) or tm.MAX_SPEED)
    su = float(tm._read(config, "sunRadius",  tm.SUN_R) or tm.SUN_R)
    bd = float(tm._read(config, "boardSize",  tm.PHYS_BOARD_SIZE) or tm.PHYS_BOARD_SIZE)
    world = tm.build_world_from_obs(
        obs, HERO, NUM_AGENTS,
        episode_seed=ep_seed, comet_speed=cs,
        ship_speed=sp, sun_radius=su, board_size=bd,
    )
    if world.is_terminal():
        return []
    macs = world.get_legal_macro_actions(HERO)
    if not macs:
        return []
    bm, _ = MCTS(net, num_simulations=EVAL_SIMS, c_puct=C_PUCT).run(world, macs)
    return tm.macro_to_env_moves(bm) if bm is not None else []

# 组装阵容
roster = [net_agent if i == HERO else opp_fn for i in range(NUM_AGENTS)]
print(f"阵容: {['net_agent' if i == HERO else OPPONENT for i in range(NUM_AGENTS)]}")

In [ ]:
# 运行对局
print("对局进行中...")
env.run(roster)

# 打印结果
final = env.steps[-1]
print(f"\n共 {len(env.steps)} 步")
for i, s in enumerate(final):
    tag = "← net" if i == HERO else f"← {OPPONENT}"
    print(f"  Player {i}: reward={s.reward:.3f}  status={s.status}  {tag}")

In [ ]:
# 渲染动画
env.render(mode="ipython", width=800, height=600)

---
## 批量评估（可选）
想快速看胜率而不看动画时用这段。

In [ ]:
from eval_orbit_wars import evaluate_net_vs_random, evaluate_net_vs_deepseek

BATCH_EPISODES = 12

wr, tr = evaluate_net_vs_random(net,   episodes=BATCH_EPISODES, num_agents=2)
wd, td = evaluate_net_vs_deepseek(net, episodes=BATCH_EPISODES, num_agents=2)

print(f"vs random   : {wr}/{tr}  ({wr/tr*100:.0f}%)")
print(f"vs deepseek : {wd}/{td}  ({wd/td*100:.0f}%)")